In [ ]:
import pandas as pd
import missingno as msno
import geopandas as gpd
from shapely.geometry import Point
import rasterio
import matplotlib.pyplot as plt
from rasterio.plot import show
import numpy as np

# 0.5 - Merge RegBL and Climate

This notebooks' objective is to merge two datasets in their `processed` versions: 

- `05_METESUISSE_climate`
- `03_OFS_regbl`

## Constants

In [ ]:
ROOT = '../../'

## Load the data

### Sunshine duration raster

In [ ]:
src = rasterio.open(f'{ROOT}data/raw/05_METESUISSE_climate/Duree_ensoleillement_1990_2020.tif')

sun_data = src.read(1)

print(src.profile)
print(src.bounds)
print(src.crs)

In [ ]:
# Mask the background values (255)
sun_data = np.ma.masked_where(sun_data == 255, sun_data)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
cax = ax.imshow(sun_data, cmap='viridis')
plt.show()

### Precipitation raster

In [ ]:
src = rasterio.open(f'{ROOT}data/raw/05_METESUISSE_climate/Precipitations_1990_2020.tif')

prec_data = src.read(1)

print(src.profile)
print(src.bounds)
print(src.crs)

In [ ]:
# Mask the background values (255)
prec_data = np.ma.masked_where(prec_data == 255, prec_data)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
cax = ax.imshow(prec_data, cmap='viridis')
plt.show()

### Temperature raster

In [ ]:
src = rasterio.open(f'{ROOT}data/raw/05_METESUISSE_climate/Temperature_1990_2020.tif')

temp_data = src.read(1)

print(src.profile)
print(src.bounds)
print(src.crs)

In [ ]:
# Mask the background values (255)
temp_data = np.ma.masked_where(temp_data == 255, temp_data)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
cax = ax.imshow(temp_data, cmap='viridis')
plt.show()

### Current dataset

In [ ]:
df_regbl = pd.read_csv(f'{ROOT}data/processed/dataset-v0.2.csv', index_col=0)

df_regbl.head()

## Convert RegBL to GeoDataFrame

In [ ]:
df_regbl = gpd.GeoDataFrame(df_regbl, geometry=gpd.points_from_xy(df_regbl["Coordonnée E"], df_regbl["Coordonnée N"]), crs='EPSG:2056')

## Overlay sunshine duration raster on RegBL to obtain sunshine duration for each building

In [ ]:
with rasterio.open(f'{ROOT}/data/raw/05_METESUISSE_climate/Duree_ensoleillement_1990_2020.tif') as src:
    sun_band = src.read(1)
    raster_crs = src.crs
    transform = src.transform

    sun_values = []
    for geom in df_regbl.geometry:
        try:
            row, col = src.index(geom.x, geom.y)
            val = sun_band[row, col]
        except IndexError:
            val = None  # if point is outside raster bounds
        sun_values.append(val)

df_regbl["Clim - Durée ensoleillement"] = sun_values

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
df_regbl.plot(column='Clim - Durée ensoleillement', ax=ax, legend=True, cmap='viridis')
plt.show()

## Overlay precipitation raster on RegBL to obtain precipitation for each building

In [ ]:
with rasterio.open(f'{ROOT}/data/raw/05_METESUISSE_climate/Precipitations_1990_2020.tif') as src:
    prec_band = src.read(1)
    raster_crs = src.crs
    transform = src.transform

    prec_values = []
    for geom in df_regbl.geometry:
        try:
            row, col = src.index(geom.x, geom.y)
            val = prec_band[row, col]
        except IndexError:
            val = None  # if point is outside raster bounds
        prec_values.append(val)

df_regbl["Clim - Précipitation"] = prec_values

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
df_regbl.plot(column='Clim - Précipitation', ax=ax, legend=True, cmap='viridis')
plt.show()

## Overlay temperature raster on RegBL to obtain temperature for each building

In [ ]:
with rasterio.open(f'{ROOT}/data/raw/05_METESUISSE_climate/Temperature_1990_2020.tif') as src:
    temp_band = src.read(1)
    raster_crs = src.crs
    transform = src.transform

    temp_values = []
    for geom in df_regbl.geometry:
        try:
            row, col = src.index(geom.x, geom.y)
            val = temp_band[row, col]
        except IndexError:
            val = None  # if point is outside raster bounds
        temp_values.append(val)

df_regbl["Clim - Température"] = temp_values

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
df_regbl.plot(column='Clim - Température', ax=ax, legend=True, cmap='viridis')
plt.show()

## Convert RegBL back to DataFrame

In [ ]:
df_regbl = pd.DataFrame(df_regbl.drop(columns='geometry'))

In [ ]:
df_regbl.head()

In [ ]:
df_regbl.shape

In [ ]:
msno.matrix(df_regbl, labels=True, fontsize=8)

## Save the data

In [ ]:
df_regbl.to_csv(f"{ROOT}data/processed/dataset-v0.3.csv")